# Buoi 2 - EDA Visual Notebook

## Thứ tự học và chạy đề xuất

- Phần 1 -> 10: chạy code theo từng phần (thực hành chính).
- Phần 11 -> 13: tư duy chuyên sâu để phân tích lỗi thực chiến.
- Phần 14 -> 15: đọc sâu ý nghĩa output và nguồn gốc từng con số.

# Buổi 2 - EDA Visual Notebook (Hướng dẫn từ số 0)

Notebook này dành cho người mới bắt đầu hoàn toàn. Mục tiêu là giúp bạn **hiểu dữ liệu trước khi train model**.

## 1) Bạn đang làm gì trong buổi này?

Bạn sẽ đi qua 4 việc chính:

1. **EDA** (khám phá dữ liệu): xem bộ ảnh có bao nhiêu ảnh, ảnh sáng/tối ra sao, nhãn có ổn không.
2. **Clean dữ liệu** (làm sạch): bỏ ảnh lỗi, ảnh thiếu nhãn, nhãn sai định dạng.
3. **Balance dữ liệu** (cân bằng): giảm lệch dữ liệu (ví dụ ảnh "normal" quá nhiều, "bright" quá ít).
4. **Augmentation** (tăng cường dữ liệu): tạo thêm ảnh biến thể và vẫn giữ nhãn bbox đúng.

---

## 2) Thuật ngữ cực cơ bản (đọc là hiểu ngay)

- **Dataset**: bộ dữ liệu (ảnh + nhãn).
- **Label**: thông tin gán cho ảnh (ở đây là vị trí biển số).
- **BBox** (*Bounding Box*): hình chữ nhật khoanh biển số.
- **YOLO label**: mỗi dòng có 5 số: `class_id x_center y_center width height` (đã chuẩn hóa 0-1).
- **EDA**: bước "khám bệnh" dữ liệu trước khi train.
- **Preprocess**: xử lý ảnh về chuẩn chung (ví dụ resize 640x640).
- **Augmentation**: tạo biến thể ảnh (sáng/tối, xoay nhẹ...) để model học tốt hơn.
- **Split train/val/test**:
  - `train`: dữ liệu để học
  - `val`: dữ liệu để canh tham số
  - `test`: dữ liệu để kiểm tra cuối cùng

---

## 3) Cấu trúc dữ liệu trong project này

- Ảnh gốc: `data/images/`
- Nhãn YOLO: `data/labels/raw/`
- Báo cáo EDA: `reports/eda/`
- Ảnh augmentation: `data/interim/augmented/`

---

## 4) Chạy nhanh toàn bộ pipeline (nếu bạn chưa chạy)

Mở terminal tại thư mục project rồi chạy:

```bash
python scripts/eda_dataset.py
python scripts/clean_balance_dataset.py
python scripts/preprocess_augment.py
```

Sau đó quay lại notebook và bấm **Run All** để xem toàn bộ kết quả trực quan.

---

## 5) Cách đọc kết quả trong notebook

- Nếu `images_with_labels` thấp: dữ liệu đang thiếu nhãn.
- Nếu `images_without_labels` cao: cần gán nhãn thêm.
- Nếu `avg_area_ratio` quá nhỏ: biển số quá bé, model sẽ khó học.
- Nếu bucket sáng/tối lệch mạnh: nên cân bằng hoặc bổ sung dữ liệu.
- Ở phần gallery augmentation: kiểm tra xem bbox có còn bám đúng biển số sau biến đổi hay không.

---

## 6) Checklist "đã ổn để qua Buổi 3"

- [ ] EDA chạy thành công, có report.
- [ ] Dữ liệu đã clean, không còn lỗi nhãn cơ bản.
- [ ] Có file cân bằng và báo cáo clean/balance.
- [ ] Augmentation sinh ảnh + file `.txt` tương ứng.
- [ ] Bạn hiểu mỗi chỉ số chính đang nói lên điều gì.

> Mẹo học nhanh: mỗi khi thấy một con số, hãy tự hỏi: "Con số này giúp mình quyết định điều gì cho bước train?"

## Phần 1 - Chuẩn bị môi trường và đường dẫn

### Bạn cần hiểu gì
- Cell này kiểm tra/cài thư viện cần thiết (`matplotlib`, `Pillow`, `opencv-python`).
- Đồng thời tạo các biến đường dẫn để các cell sau dùng chung, tránh viết lặp.

### Ví dụ chuyên ngành
- Trước khi train detector, bạn cần xác nhận môi trường có đủ thư viện xử lý ảnh (`cv2`, `Pillow`) và trực quan (`matplotlib`), tương tự bước kiểm tra dependency trong pipeline MLOps.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: `ModuleNotFoundError` (`matplotlib`, `PIL`, `cv2`).
  **Tự sửa**: chạy lại cell này để notebook tự cài; nếu vẫn lỗi, restart kernel rồi chạy lại từ đầu.
- **Lỗi**: đường dẫn `project_root` sai.
  **Tự sửa**: kiểm tra dòng print `project_root`, đảm bảo đang trỏ đúng thư mục project.

In [ ]:
from pathlib import Path
import json
import math
import importlib
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Dang cai '{pip_name}' cho kernel hien tai...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("matplotlib", "matplotlib")
ensure_package("PIL", "Pillow")
ensure_package("cv2", "opencv-python")

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

project_root = Path.cwd().resolve()
if project_root.name == "docs":
    project_root = project_root.parent

report_json_path = project_root / "reports" / "eda" / "dataset_report.json"
clean_report_path = project_root / "reports" / "eda" / "clean_balance_report.json"
preview_dir = project_root / "reports" / "eda" / "preview_samples"
augmented_dir = project_root / "data" / "interim" / "augmented"

print(f"project_root: {project_root}")
print(f"report_json_path: {report_json_path}")
print(f"clean_report_path: {clean_report_path}")
print(f"preview_dir: {preview_dir}")
print(f"augmented_dir: {augmented_dir}")

## Phần 2 - Nạp báo cáo EDA từ file JSON

### Bạn cần hiểu gì
- Cell này đảm bảo có file `dataset_report.json` rồi mới đọc.
- Nếu chưa có, notebook sẽ tự chạy script EDA để tạo mới.

### Ví dụ đời thường
- Giống mở bảng điểm: nếu chưa có điểm, hệ thống phải chấm xong mới mở cho bạn xem.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: thiếu `dataset_report.json`.
  **Tự sửa**: chạy `python scripts/eda_dataset.py` rồi chạy lại cell.
- **Lỗi**: `RuntimeError` khi auto chạy script.
  **Tự sửa**: đọc log `STDERR` ngay dưới cell để biết thiếu thư viện hay sai path.

In [ ]:
if not report_json_path.exists():
    candidate_images_dirs = [
        project_root / "images",
        project_root / "data" / "images",
        project_root / "data" / "raw",
    ]
    candidate_labels_dirs = [
        project_root / "labels",
        project_root / "data" / "labels" / "raw",
        project_root / "data" / "labels",
    ]

    images_dir = next((p for p in candidate_images_dirs if p.exists()), None)
    labels_dir = next((p for p in candidate_labels_dirs if p.exists()), None)

    if images_dir is None or labels_dir is None:
        raise FileNotFoundError(
            "Khong tim thay du lieu EDA. Da thu: images/, data/images/, data/raw/ va labels/."
        )

    print("Khong tim thay dataset_report.json -> tu dong tao moi...")
    print(f"images_dir: {images_dir}")
    print(f"labels_dir: {labels_dir}")

    cmd = [
        sys.executable,
        str(project_root / "scripts" / "eda_dataset.py"),
        "--images-dir",
        str(images_dir),
        "--labels-dir",
        str(labels_dir),
        "--output-dir",
        str(project_root / "reports" / "eda"),
        "--num-preview",
        "12",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== STDOUT ===")
        print(result.stdout)
        print("=== STDERR ===")
        print(result.stderr)
        raise RuntimeError("eda_dataset.py chay that bai, xem log o tren.")

report = json.loads(report_json_path.read_text(encoding="utf-8"))
report

## Phần 3 - Đọc các chỉ số EDA tổng quan

### Bạn cần hiểu gì
- Cell này in các chỉ số chính: số ảnh, số box, kích thước ảnh, độ sáng.
- Đây là "ảnh chụp sức khỏe" của dataset trước khi train.

### Ví dụ chuyên ngành
- Đây là bước dataset profiling: bạn đọc `images_with_labels`, `avg_area_ratio`, `brightness_distribution` để dự đoán rủi ro fail ở stage detector/OCR trước khi tốn tài nguyên train.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: chỉ số ra 0 bất thường (ví dụ `images_with_labels = 0`).
  **Tự sửa**: kiểm tra lại thư mục ảnh/nhãn và chạy lại EDA.
- **Lỗi**: `NameError` biến `report`.
  **Tự sửa**: chạy lại các cell trước theo thứ tự từ trên xuống.

In [ ]:
overview = report.get("dataset_overview", {})
size_stats = report.get("image_size_stats", {})
plate_stats = report.get("plate_box_stats", {})
brightness = report.get("brightness_distribution", {})

print("=== OVERVIEW ===")
for k, v in overview.items():
    print(f"{k}: {v}")

print("\n=== IMAGE SIZE STATS ===")
for k, v in size_stats.items():
    print(f"{k}: {v}")

print("\n=== PLATE BOX STATS ===")
for k, v in plate_stats.items():
    print(f"{k}: {v}")

print("\n=== BRIGHTNESS DISTRIBUTION ===")
for k, v in brightness.items():
    print(f"{k}: {v}")

## Phần 4 - Vẽ biểu đồ độ sáng ảnh

### Bạn cần hiểu gì
- Biểu đồ giúp bạn thấy dữ liệu có bị nghiêng về ảnh tối hay ảnh sáng không.
- Phân bố ánh sáng ảnh hưởng trực tiếp đến khả năng đọc biển số.

### Ví dụ chuyên ngành
- Nếu bucket `dark` quá ít, detector sẽ học representation lệch về điều kiện sáng chuẩn và giảm recall khi suy luận trên camera đêm.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: biểu đồ trống hoặc toàn số 0.
  **Tự sửa**: kiểm tra biến `brightness` đã được tạo ở phần EDA tổng quan chưa.
- **Lỗi**: lỗi hiển thị font tiếng Việt.
  **Tự sửa**: vẫn có thể dùng biểu đồ bình thường; nếu cần, cấu hình font matplotlib sau.

In [ ]:
# Ve bieu do phan bo anh toi/normal/sang
labels = ["dark", "normal", "bright"]
values = [brightness.get(k, 0) for k in labels]

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, values)
plt.title("Brightness Distribution")
plt.xlabel("Lighting")
plt.ylabel("Image count")
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, val, str(val), ha="center", va="bottom")
plt.show()

## Phần 5 - Xem gallery ảnh preview box từ EDA

### Bạn cần hiểu gì
- Đây là kiểm tra trực quan: bbox có đang bao đúng biển số không.
- Nếu box lệch nhiều, train model sẽ học sai và độ chính xác giảm.

### Ví dụ chuyên ngành
- Với object detection, nếu bbox GT lệch khỏi biển số thì loss sẽ tối ưu sai mục tiêu và mô hình hội tụ về nhãn nhiễu (label noise).

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: `preview_samples` không tồn tại.
  **Tự sửa**: chạy lại `python scripts/eda_dataset.py` để tạo preview.
- **Lỗi**: có ảnh nhưng không thấy box.
  **Tự sửa**: kiểm tra file label gốc có box hợp lệ hay không.

In [ ]:
# Hien thi gallery anh preview box
if not preview_dir.exists():
    raise FileNotFoundError("Khong tim thay preview_samples. Hay chay scripts/eda_dataset.py truoc.")

image_paths = sorted(preview_dir.glob("*.jpg"))
if not image_paths:
    print("Khong co anh preview nao de hien thi.")
else:
    max_show = min(12, len(image_paths))
    cols = 3
    rows = math.ceil(max_show / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes

    axes_flat = axes.ravel() if hasattr(axes, "ravel") else axes

    for i in range(max_show):
        img = Image.open(image_paths[i])
        axes_flat[i].imshow(img)
        axes_flat[i].set_title(image_paths[i].name, fontsize=9)
        axes_flat[i].axis("off")

    for j in range(max_show, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()
    plt.show()

## Phần 6 - Nạp file kết quả clean/balance

### Bạn cần hiểu gì
- Cell này đọc file `clean_balance_report.json` vào biến `clean_report`.
- Nếu file chưa có, notebook báo lỗi để nhắc bạn chạy script trước.

### Ví dụ chuyên ngành
- `clean_balance_report.json` là artifact hậu kiểm dữ liệu: chứa thống kê mẫu bị loại, mẫu giữ lại và mức cân bằng để phục vụ audit dữ liệu train.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: thiếu `clean_balance_report.json`.
  **Tự sửa**: chạy `python scripts/clean_balance_dataset.py`.
- **Lỗi**: đọc file nhưng key bị thiếu.
  **Tự sửa**: xóa report cũ, chạy lại script để tạo report mới đúng schema.

In [ ]:
# Doc bao cao clean + balance
if not clean_report_path.exists():
    raise FileNotFoundError(
        "Khong tim thay clean_balance_report.json. Hay chay scripts/clean_balance_dataset.py truoc."
    )

clean_report = json.loads(clean_report_path.read_text(encoding="utf-8"))
clean_report

## Phần 7 - Đọc báo cáo clean/balance bằng chữ

### Bạn cần hiểu gì
- Đây là phần giải thích con số tổng hợp: giữ lại bao nhiêu ảnh, loại bao nhiêu ảnh, vì sao bị loại.
- Bạn cần tập thói quen đọc con số để ra quyết định: có cần gán nhãn thêm hay không.

### Ví dụ chuyên ngành
- Trong data-centric AI, bạn cần định lượng chính xác tỷ lệ dữ liệu bị loại vì lỗi label/ảnh hỏng để ước lượng tác động đến generalization của model.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: `images_kept` thấp bất thường.
  **Tự sửa**: kiểm tra chất lượng label (format YOLO, giá trị 0-1).
- **Lỗi**: report có số âm/không hợp lý.
  **Tự sửa**: chạy lại script clean với đúng thư mục dữ liệu.

In [ ]:
# Tong quan ket qua clean/balance
print("=== CLEAN/BALANCE OVERVIEW ===")
print(f"images_total: {clean_report.get('images_total', 0)}")
print(f"images_kept: {clean_report.get('images_kept', 0)}")
print(f"balanced_images_total: {clean_report.get('balanced_images_total', 0)}")

print("\n=== DROPPED ===")
for k, v in clean_report.get("dropped", {}).items():
    print(f"{k}: {v}")

print("\n=== BUCKETS BEFORE BALANCE ===")
for k, v in clean_report.get("brightness_buckets_before_balance", {}).items():
    print(f"{k}: {v}")

print("\n=== OUTPUTS ===")
for k, v in clean_report.get("outputs", {}).items():
    print(f"{k}: {v}")

## Phần 8 - So sánh trước và sau khi cân bằng dữ liệu

### Bạn cần hiểu gì
- Dữ liệu lệch (ví dụ ảnh `normal` quá nhiều) làm model học thiên vị.
- Biểu đồ cột `before` vs `after` cho bạn thấy việc cân bằng đã kéo các nhóm về gần nhau chưa.

### Ví dụ chuyên ngành
- Cân bằng bucket ánh sáng là một dạng distribution alignment, giúp giảm domain bias giữa dữ liệu train và bối cảnh triển khai thực tế.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: cột `after(balance)` bằng 0.
  **Tự sửa**: xem lại `target_per_bucket_after_balance` trong report, có thể một bucket đang rỗng.
- **Lỗi**: biểu đồ không thay đổi nhiều.
  **Tự sửa**: kiểm tra bạn đã chạy script clean/balance mới nhất chưa.

In [ ]:
# Truc quan phan bo anh sang truoc va sau can bang
before = clean_report.get("brightness_buckets_before_balance", {})
labels_cb = ["dark", "normal", "bright"]
before_vals = [before.get(k, 0) for k in labels_cb]

target = int(clean_report.get("target_per_bucket_after_balance", 0))
after_vals = [target if before.get(k, 0) > 0 else 0 for k in labels_cb]

x = range(len(labels_cb))
width = 0.38

plt.figure(figsize=(8, 4))
plt.bar([i - width / 2 for i in x], before_vals, width=width, label="before")
plt.bar([i + width / 2 for i in x], after_vals, width=width, label="after(balance)")
plt.xticks(list(x), labels_cb)
plt.title("Brightness Buckets: Before vs After Balance")
plt.xlabel("Bucket")
plt.ylabel("Image count")
plt.legend()
plt.tight_layout()
plt.show()

## Phần 9 - Xem ảnh augmentation và bbox sau biến đổi

### Bạn cần hiểu gì
- Sau khi augment ảnh, nhãn bbox phải đi theo đúng vị trí mới của biển số.
- Cell này giúp bạn nhìn trực tiếp ảnh augment và khung bbox để kiểm tra chất lượng.

### Ví dụ chuyên ngành
- Khi augmentation có phép quay/biến đổi hình học, label YOLO phải được transform cùng ma trận; nếu không sẽ tạo sample nhiễu làm giảm cả precision lẫn recall.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: không có ảnh trong `data/interim/augmented`.
  **Tự sửa**: chạy `python scripts/preprocess_augment.py`.
- **Lỗi**: có ảnh augment nhưng thiếu `.txt`.
  **Tự sửa**: đảm bảo script chạy với `labels-dir` hợp lệ (mặc định hiện đã có).

In [ ]:
# Gallery augmentation + bbox da dong bo
if not augmented_dir.exists():
    raise FileNotFoundError(
        "Khong tim thay thu muc augmented. Hay chay scripts/preprocess_augment.py truoc."
    )

aug_image_paths = sorted(augmented_dir.rglob("*.jpg"))
if not aug_image_paths:
    print("Khong co anh augmentation nao de hien thi.")
else:
    max_show = min(9, len(aug_image_paths))
    cols = 3
    rows = math.ceil(max_show / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes_flat = axes.ravel() if hasattr(axes, "ravel") else [axes]

    for i in range(max_show):
        img_path = aug_image_paths[i]
        img = Image.open(img_path).convert("RGB")
        ax = axes_flat[i]
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=9)
        ax.axis("off")

        label_path = img_path.with_suffix(".txt")
        if label_path.exists():
            w, h = img.size
            for line in label_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, xc, yc, bw, bh = parts
                xc = float(xc)
                yc = float(yc)
                bw = float(bw)
                bh = float(bh)

                x1 = (xc - bw / 2.0) * w
                y1 = (yc - bh / 2.0) * h
                rect = Rectangle(
                    (x1, y1),
                    bw * w,
                    bh * h,
                    linewidth=1.5,
                    edgecolor="lime",
                    facecolor="none",
                )
                ax.add_patch(rect)

    for j in range(max_show, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()
    plt.show()

## Phần 10 - Kiểm tra split train/val/test

### Bạn cần hiểu gì
- `train/val/test` là 3 nhóm dữ liệu cho 3 mục đích khác nhau khi học máy.
- Cell này chỉ đếm nhanh số ảnh mỗi nhóm để bạn kiểm tra có bị lệch bất thường không.

### Ví dụ chuyên ngành
- Trong đánh giá mô hình, `train/val/test` phải tách độc lập để tránh data leakage; nếu leakage xảy ra, metric cao giả tạo nhưng fail khi deploy.

### Lỗi hay gặp + cách tự sửa
- **Lỗi**: thiếu `train.txt/val.txt/test.txt`.
  **Tự sửa**: chạy `python scripts/split_dataset.py` để tạo split.
- **Lỗi**: số lượng split lệch bất thường.
  **Tự sửa**: kiểm tra lại tỉ lệ `train/val/test` và seed khi split.

In [ ]:
# Kiem tra nhanh split train/val/test
split_dir = project_root / "data" / "splits"
for split_name in ["train.txt", "val.txt", "test.txt"]:
    split_path = split_dir / split_name
    if not split_path.exists():
        print(f"{split_name}: (chua co)")
        continue
    lines = [ln for ln in split_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    print(f"{split_name}: {len(lines)} images")

## Phần 11 - Góc chuyên sâu AI/CV

Phần này dành cho góc nhìn thực chiến khi bạn đã chạy xong EDA + clean/balance + augmentation.

### 5 case study thực chiến

### Case 1 - `mAP50` ổn nhưng biển số nhỏ vẫn miss nhiều
- **Triệu chứng**: metric tổng đẹp, nhưng ảnh xa/camera cao bị bỏ sót.
- **Nguyên nhân gốc**: object quá nhỏ so với ảnh, feature map mất chi tiết.
- **Hướng xử lý**:
  - tăng `imgsz` khi train/infer (nếu tài nguyên cho phép),
  - ưu tiên dữ liệu có biển số nhỏ trong train,
  - đánh giá riêng nhóm small objects thay vì chỉ nhìn mAP chung.

### Case 2 - OCR sai ký tự giống nhau (`0/O`, `1/I`, `5/S`, `8/B`)
- **Triệu chứng**: detector đúng box nhưng chuỗi biển số sai 1-2 ký tự.
- **Nguyên nhân gốc**: ảnh crop mờ, nghiêng, hoặc OCR thiếu ngữ cảnh ký tự.
- **Hướng xử lý**:
  - thêm bước rectify/denoise trước OCR,
  - fallback OCR model khi confidence thấp,
  - postprocess theo regex biển số VN + luật sửa nhầm ký tự theo vị trí.

### Case 3 - Model chạy tốt ban ngày, kém mạnh ban đêm
- **Triệu chứng**: accuracy giảm mạnh ở bucket `dark`.
- **Nguyên nhân gốc**: lệch phân bố ánh sáng trong dữ liệu train.
- **Hướng xử lý**:
  - cân bằng lại bucket ánh sáng,
  - augment sáng/tối có kiểm soát,
  - report metric theo từng bucket (`dark/normal/bright`).

### Case 4 - Loss train giảm nhưng metric val không tăng
- **Triệu chứng**: train đẹp nhưng val dậm chân hoặc giảm.
- **Nguyên nhân gốc**: overfit, label nhiễu, split chưa đại diện.
- **Hướng xử lý**:
  - kiểm tra lại chất lượng label ở val,
  - giảm độ mạnh augmentation gây méo semantics,
  - dùng early stopping + regularization + seed cố định.

### Case 5 - End-to-end sai nhiều dù detector/OCR riêng lẻ khá tốt
- **Triệu chứng**: từng stage tốt nhưng output cuối vẫn sai nhiều.
- **Nguyên nhân gốc**: lỗi tích lũy giữa các stage (detect -> crop -> OCR -> rule).
- **Hướng xử lý**:
  - log trung gian cho từng stage,
  - tách lỗi theo nhóm: miss detect / crop lệch / OCR nhầm / rule sửa sai,
  - tối ưu theo metric cuối: `plate_accuracy`, không chỉ metric stage riêng.

## Phần 12 - Checklist quyết định khi metric xấu

Dùng checklist này theo thứ tự để tránh sửa lan man.

1. **Xác định metric nào đang xấu**
   - Detection (`mAP`, recall), OCR (`CER/WER`), hay end-to-end (`plate_accuracy`).
2. **Khoanh vùng theo điều kiện dữ liệu**
   - ban ngày/ban đêm, gần/xa, góc nghiêng, mờ/nhiễu.
3. **Kiểm tra dữ liệu trước model**
   - nhãn đúng không, dữ liệu có lệch phân bố không, có nhiều ảnh lỗi không.
4. **Kiểm tra pipeline trung gian**
   - box detect có ôm đúng biển số không, crop có bị cắt mất ký tự không.
5. **Mới tối ưu hyperparameters/model**
   - sau khi xác nhận dữ liệu và pipeline ổn.
6. **So sánh bằng cùng điều kiện**
   - cùng split, cùng seed, cùng tập test để kết luận công bằng.

> Quy tắc vàng: nếu dữ liệu sai, model mạnh mấy cũng không cứu được.

## Phần 13 - Flow debug: detector vs OCR vs dữ liệu

```text
Bắt đầu -> Plate accuracy giảm
   |
   +-- B1: Kiểm tra detector
   |      - Box có bám đúng biển số không?
   |      - Nếu miss nhiều: ưu tiên sửa dữ liệu detect + tuning detect
   |
   +-- B2: Nếu detector ổn -> kiểm tra OCR
   |      - Crop có rõ, đủ ký tự, ít nghiêng không?
   |      - Nếu OCR sai nhiều: tăng preprocess OCR + postprocess rule
   |
   +-- B3: Nếu detector & OCR đều "ổn" nhưng E2E vẫn kém
   |      - Kiểm tra logic ghép stage, threshold, confidence gate
   |      - Kiểm tra rule hậu xử lý có làm hỏng output đúng không
   |
   +-- B4: Soi lại dữ liệu gốc
          - Lệch sáng/tối? nhiều ảnh mờ? label nhiễu?
          - Nếu có: clean + rebalance + augment đúng mục tiêu
```

### Mẹo thực chiến
- Luôn lưu log 10-20 mẫu lỗi điển hình sau mỗi lần train.
- Mỗi lần chỉ thay đổi **1 nhóm yếu tố** (data hoặc model hoặc postprocess) để biết chính xác thứ gì tạo ra cải thiện.
- Báo cáo kết quả theo nhóm điều kiện (dark/normal/bright, near/far), không chỉ một con số tổng.

## Phần 14 - Cách hiểu đầu ra của từng cell code

Dưới đây là "bản dịch" output theo thứ tự chạy code trong notebook.

### Cell code 1 - Setup môi trường và đường dẫn
- **Output bạn sẽ thấy**: `project_root`, `report_json_path`, `clean_report_path`, `preview_dir`, `augmented_dir`.
- **Ý nghĩa**:
  - xác nhận notebook đang trỏ đúng thư mục project,
  - xác nhận các đường dẫn artifact đúng trước khi xử lý.
- **Kết luận nhanh**:
  - nếu path đúng => có thể chạy các cell sau,
  - nếu path sai => sửa cwd hoặc mở notebook từ đúng project.

### Cell code 2 - Nạp/tạo report EDA
- **Output bạn sẽ thấy**: thông báo auto-generate report (nếu chưa có), sau đó in object `report`.
- **Ý nghĩa**:
  - `report` là dữ liệu gốc cho toàn bộ phân tích EDA phía sau.
- **Kết luận nhanh**:
  - load được `report` => pipeline EDA hoạt động,
  - lỗi ở đây => các cell thống kê/biểu đồ sau sẽ fail theo.

### Cell code 3 - In thống kê EDA tổng quan
- **Output bạn sẽ thấy**: 4 nhóm: `OVERVIEW`, `IMAGE SIZE STATS`, `PLATE BOX STATS`, `BRIGHTNESS DISTRIBUTION`.
- **Ý nghĩa**:
  - đo chất lượng và phân bố dữ liệu trước train.
- **Kết luận nhanh**:
  - `images_with_labels` thấp => thiếu nhãn,
  - `avg_area_ratio` quá nhỏ => object nhỏ khó detect,
  - bucket sáng/tối lệch => dễ bias domain.

### Cell code 4 - Biểu đồ brightness
- **Output bạn sẽ thấy**: bar chart `dark/normal/bright`.
- **Ý nghĩa**:
  - trực quan mức mất cân bằng ánh sáng.
- **Kết luận nhanh**:
  - chênh lệch lớn => nên clean/balance hoặc augment theo ánh sáng.

### Cell code 5 - Gallery preview box EDA
- **Output bạn sẽ thấy**: lưới ảnh có bbox và tên file.
- **Ý nghĩa**:
  - kiểm tra nhanh chất lượng nhãn detect bằng mắt.
- **Kết luận nhanh**:
  - box bám đúng biển số => nhãn đáng tin hơn,
  - box lệch/sai => cần sửa label trước train.

### Cell code 6 - Nạp clean/balance report
- **Output bạn sẽ thấy**: object `clean_report`.
- **Ý nghĩa**:
  - chứa kết quả làm sạch + cân bằng dữ liệu.
- **Kết luận nhanh**:
  - load được report => bước clean/balance chạy OK.

### Cell code 7 - In tổng quan clean/balance
- **Output bạn sẽ thấy**: `images_total`, `images_kept`, `balanced_images_total`, `dropped`, `outputs`.
- **Ý nghĩa**:
  - định lượng dữ liệu bị loại và dữ liệu còn lại để train.
- **Kết luận nhanh**:
  - nếu drop quá cao => cần điều tra chất lượng nhãn/ảnh,
  - nếu balanced quá thấp => bucket đang rất lệch.

### Cell code 8 - Biểu đồ before vs after balance
- **Output bạn sẽ thấy**: 2 cột mỗi bucket (`before`, `after(balance)`).
- **Ý nghĩa**:
  - kiểm chứng tác động của bước cân bằng phân bố.
- **Kết luận nhanh**:
  - cột `after` gần đều hơn => balance có hiệu quả.

### Cell code 9 - Gallery augmentation + bbox
- **Output bạn sẽ thấy**: ảnh augment và bbox tương ứng.
- **Ý nghĩa**:
  - xác nhận transform ảnh và transform label đồng bộ.
- **Kết luận nhanh**:
  - bbox vẫn ôm đúng biển số => augmentation hợp lệ cho detection,
  - bbox lệch => cần sửa logic transform nhãn.

### Cell code 10 - Kiểm tra split train/val/test
- **Output bạn sẽ thấy**: số ảnh trong `train.txt`, `val.txt`, `test.txt`.
- **Ý nghĩa**:
  - xác nhận dữ liệu đã được chia tập để train/eval đúng quy trình.
- **Kết luận nhanh**:
  - thiếu file split => chưa sẵn sàng train,
  - split quá lệch => rà lại script split và ratio.

---

## Cách đọc output theo tư duy debug nhanh

1. **Cell 1-2 fail** -> lỗi môi trường/path/artifact.
2. **Cell 3-5 bất thường** -> lỗi chất lượng dữ liệu/nhãn.
3. **Cell 6-8 bất thường** -> lỗi clean/balance hoặc lệch phân bố nặng.
4. **Cell 9 bất thường** -> lỗi đồng bộ bbox khi augmentation.
5. **Cell 10 bất thường** -> lỗi chuẩn bị dữ liệu train/eval.

Dùng thứ tự này để khoanh vùng lỗi nhanh thay vì debug toàn pipeline cùng lúc.

## Phần 15 - Vì sao ra những con số đó? (Giải thích gốc công thức)

Phần này giúp bạn hiểu **nguồn gốc từng chỉ số** thay vì chỉ đọc kết quả.

### 1) Nhóm `dataset_overview`

- `images_total`
  - **Tính từ đâu**: số file ảnh script quét được trong `images_dir` với đuôi hợp lệ (`jpg/png/...`).
  - **Vì sao tăng/giảm**:
    - tăng khi thêm ảnh vào thư mục,
    - giảm khi ảnh hỏng không đọc được hoặc lọc nhầm đường dẫn.

- `images_with_labels`
  - **Tính từ đâu**: số ảnh có ít nhất 1 bbox hợp lệ trong file label tương ứng.
  - **Vì sao tăng/giảm**:
    - giảm khi map ảnh-nhãn sai thư mục,
    - giảm khi label rỗng/sai format,
    - tăng khi sửa đúng label và đúng đường dẫn.

- `images_without_labels = images_total - images_with_labels`
  - **Ý nghĩa**: phần dữ liệu detector không học được target.
  - **Vì sao cao**: thiếu file `.txt`, nhãn sai chuẩn YOLO, hoặc mismatch tên file.

- `total_boxes`
  - **Tính từ đâu**: tổng số dòng bbox hợp lệ qua toàn bộ file label.
  - **Vì sao tăng/giảm**:
    - tăng khi ảnh có nhiều biển/label đầy đủ,
    - giảm khi label bị lỗi, bị bỏ qua khi parse.

- `avg_boxes_per_image = total_boxes / images_total`
  - **Ý nghĩa**: mật độ object trung bình mỗi ảnh.
  - **Vì sao lệch**: dataset có nhiều ảnh không biển hoặc nhiều ảnh nhiều biển.

---

### 2) Nhóm `image_size_stats`

- `avg_width`, `avg_height`
  - **Tính từ đâu**: trung bình kích thước ảnh thật đọc bằng OpenCV.
  - **Vì sao thay đổi**: dataset trộn nhiều nguồn camera khác nhau.

- `min_*`, `max_*`
  - **Ý nghĩa**: biên kích thước nhỏ nhất/lớn nhất.
  - **Vì sao quan trọng**: biên quá rộng thường làm train khó ổn định nếu preprocess không chuẩn.

---

### 3) Nhóm `plate_box_stats`

- `area_ratio` mỗi box = `bbox_width_norm * bbox_height_norm`
- `avg_area_ratio`, `min_area_ratio`, `max_area_ratio`
  - **Tính từ đâu**: thống kê trên tất cả bbox hợp lệ.
  - **Vì sao nhỏ**: biển số chiếm ít pixel (ảnh xa/góc rộng).
  - **Hệ quả**: object nhỏ -> detector khó học hơn, recall dễ giảm.

---

### 4) Nhóm `brightness_distribution`

Script gán nhãn độ sáng theo mean grayscale:
- `dark` nếu mean < 70
- `normal` nếu 70 <= mean <= 170
- `bright` nếu mean > 170

**Vì sao ra số đó**:
- ảnh chụp thiếu sáng -> dồn vào `dark`,
- ảnh đủ sáng đa số -> dồn vào `normal`,
- ảnh cháy sáng/gắt nắng -> tăng `bright`.

**Lưu ý chuyên môn**: đây là heuristic ngưỡng nhanh, không phải chỉ số photometric tuyệt đối.

---

### 5) Nhóm clean/balance (`clean_balance_report.json`)

- `images_kept`
  - **Tính từ đâu**: ảnh đọc được + có label tồn tại + có ít nhất 1 bbox hợp lệ.
- `dropped.missing_labels`
  - ảnh không tìm thấy file label map đúng.
- `dropped.bad_images`
  - ảnh OpenCV đọc lỗi (`cv2.imread` trả `None`).
- `dropped.invalid_or_empty_labels`
  - có file label nhưng không có bbox hợp lệ.
- `balanced_images_total`
  - bằng tổng số ảnh sau khi undersample mỗi bucket sáng/tối về `target_per_bucket_after_balance`.

**Vì sao `balanced_images_total` có thể nhỏ**:
- vì bucket nhỏ nhất quyết định trần lấy mẫu cho các bucket khác.

---

### 6) Nhóm split train/val/test

Số dòng trong `train.txt`, `val.txt`, `test.txt` phụ thuộc:
- tổng ảnh đầu vào khi split,
- tỉ lệ cấu hình (`train/val/test`),
- cách làm tròn số nguyên.

Nếu bạn thấy lệch nhỏ 1-2 ảnh là bình thường do quy tắc làm tròn.

---

## Công thức suy luận nguyên nhân nhanh

- `images_with_labels` thấp + `missing_labels` cao -> lỗi map đường dẫn ảnh-nhãn.
- `total_boxes` thấp nhưng `images_with_labels` cao -> nhiều label có ít box hoặc box lỗi.
- `avg_area_ratio` rất thấp + ảnh độ phân giải cao -> biển nhỏ thật, cần chiến lược small-object.
- `dark` quá cao + metric đêm thấp -> thiếu dữ liệu sáng chuẩn hoặc preprocess ánh sáng chưa đủ.

Dùng các quy tắc này để đi từ **con số -> nguyên nhân -> hành động sửa**.

---

## Phần 16 - Ý nghĩa thuật toán và công thức EDA trong dự án này

Buổi 2 không train model ngay. Buổi 2 trả lời câu hỏi nền tảng:

> Dữ liệu biển số của mình có đủ sạch, đủ cân bằng và đủ đáng tin để đưa sang Buổi 3 train YOLO không?

Pipeline của dự án có thể hiểu như sau:

$$
\text{Dữ liệu ảnh + nhãn} \rightarrow \text{EDA} \rightarrow \text{Clean} \rightarrow \text{Balance} \rightarrow \text{Augment} \rightarrow \text{Train detector ở Buổi 3}
$$

Nếu Buổi 2 làm chưa tốt, Buổi 3 có thể gặp các lỗi như:

- YOLO học sai vì bbox label lệch.
- Model bỏ sót biển số nhỏ vì `area_ratio` quá thấp.
- Model kém trong ảnh tối/sáng gắt vì phân bố brightness lệch.
- Metrics val/test không đáng tin vì split dữ liệu sai hoặc mất cân bằng.

### 1) Thuật toán quét dataset

Script EDA duyệt qua toàn bộ ảnh, tìm file label tương ứng, đọc kích thước ảnh và đọc bbox YOLO.

Với mỗi ảnh:

$$
image_i \rightarrow (width_i, height_i, labels_i)
$$

Ý nghĩa trong dự án:

- Biết có bao nhiêu ảnh dùng được.
- Biết bao nhiêu ảnh thiếu nhãn.
- Biết bbox biển số có hợp lệ hay không.
- Biết ảnh có kích thước quá nhỏ/quá lớn hay không.

### 2) Công thức YOLO label

Một dòng label YOLO có dạng:

$$
(class\_id, x_c, y_c, w, h)
$$

Trong đó $x_c, y_c, w, h$ đã được chuẩn hóa trong khoảng $[0, 1]$.

Chuyển về pixel thật:

$$
x_1 = (x_c - \frac{w}{2}) \times W
$$

$$
y_1 = (y_c - \frac{h}{2}) \times H
$$

$$
x_2 = (x_c + \frac{w}{2}) \times W
$$

$$
y_2 = (y_c + \frac{h}{2}) \times H
$$

Ý nghĩa trong dự án:

- Công thức này giúp vẽ bbox lên ảnh để kiểm tra nhãn bằng mắt.
- Nếu bbox vẽ lệch, có thể label sai hoặc transform augmentation chưa đồng bộ.

### 3) Area ratio - biển số chiếm bao nhiêu phần ảnh?

Vì YOLO label đã chuẩn hóa, diện tích tương đối của bbox là:

$$
area\_ratio = w \times h
$$

Ý nghĩa trong dự án:

- `area_ratio` nhỏ: biển số rất bé trong ảnh, detector khó học hơn.
- `area_ratio` lớn bất thường: bbox có thể khoanh quá rộng hoặc nhãn sai.
- `avg_area_ratio` giúp ước lượng độ khó của bài toán detection.

Trong report hiện tại:

$$
avg\_area\_ratio \approx 0.0432
$$

Tức là trung bình biển số chiếm khoảng 4.32% diện tích ảnh. Đây là object tương đối nhỏ, nên cần chú ý resize, augmentation và chất lượng bbox.

### 4) Brightness distribution - phân bố độ sáng

EDA phân loại ảnh theo độ sáng trung bình grayscale:

$$
brightness = \frac{1}{W \times H}\sum_{x=1}^{W}\sum_{y=1}^{H} I(x,y)
$$

Quy tắc bucket trong notebook:

- `dark` nếu brightness thấp.
- `normal` nếu brightness trung bình.
- `bright` nếu brightness cao.

Ý nghĩa trong dự án:

- Nếu ảnh `normal` quá nhiều nhưng `dark`/`bright` quá ít, model dễ bias theo điều kiện đủ sáng.
- Khi gặp ảnh ban đêm hoặc chói sáng, model có thể detect kém.

Trong report hiện tại:

$$
dark = 404,\quad normal = 4143,\quad bright = 31
$$

Phân bố này lệch mạnh về `normal`, nên bước balance/augmentation là cần thiết.

### 5) Balance dữ liệu

Mục tiêu balance là giảm chênh lệch giữa các nhóm dữ liệu.

Nếu số ảnh từng bucket là:

$$
N_{dark}, N_{normal}, N_{bright}
$$

Một cách cân bằng đơn giản là chọn số mẫu mỗi bucket gần bằng:

$$
N_{target} = \min(N_{dark}, N_{normal}, N_{bright})
$$

Ý nghĩa trong dự án:

- Giúp model không chỉ học tốt trên nhóm ảnh phổ biến nhất.
- Giảm rủi ro model kém khi gặp điều kiện ánh sáng hiếm.
- Tuy nhiên undersampling có thể làm mất bớt dữ liệu, nên cần cân bằng giữa “đều” và “đủ nhiều”.

### 6) Augmentation

Augmentation tạo ảnh biến thể như sáng hơn, xoay nhẹ, resize. Về mặt ý tưởng:

$$
(image, bbox) \rightarrow T(image, bbox)
$$

Trong đó $T$ là phép biến đổi.

Điểm quan trọng:

- Ảnh biến đổi thì bbox cũng phải biến đổi theo.
- Nếu ảnh xoay/resize mà bbox không đổi đúng, model sẽ học nhãn sai.

Vì vậy gallery augmentation trong notebook có ý nghĩa rất lớn: nó giúp kiểm tra bằng mắt xem bbox sau biến đổi còn bám đúng biển số không.

In [ ]:
# Sơ đồ luồng hoạt động Buổi 2.
# Cell này chỉ vẽ khi chạy, không nhúng sẵn ảnh vào notebook.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

eda_steps = [
    ("1. Quét ảnh", "Tìm file ảnh\nvà kích thước"),
    ("2. Đọc label", "Kiểm tra file .txt\nchuẩn YOLO"),
    ("3. Thống kê", "Số ảnh, số bbox,\nkích thước, brightness"),
    ("4. Clean", "Bỏ ảnh lỗi,\nlabel rỗng/sai"),
    ("5. Balance", "Giảm lệch dark /\nnormal / bright"),
    ("6. Augment", "Tạo biến thể ảnh\n+ cập nhật bbox"),
    ("7. Split", "Tạo train / val / test\ncho Buổi 3"),
]

fig, ax = plt.subplots(figsize=(16, 4))
ax.set_xlim(0, len(eda_steps))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, (title, body) in enumerate(eda_steps):
    x = idx + 0.08
    color = "#EEF5FF" if idx < 3 else "#F2FFF0"
    box = FancyBboxPatch(
        (x, 0.25),
        0.78,
        0.5,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        linewidth=1.5,
        edgecolor="#4C78A8",
        facecolor=color,
    )
    ax.add_patch(box)
    ax.text(x + 0.39, 0.58, title, ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(x + 0.39, 0.42, body, ha="center", va="center", fontsize=9)
    if idx < len(eda_steps) - 1:
        ax.annotate("", xy=(idx + 1.03, 0.5), xytext=(idx + 0.88, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))

ax.set_title("Luồng hoạt động EDA - Clean - Balance - Augment trong Buổi 2", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Biểu đồ tổng hợp chất lượng dataset.
# Ưu tiên dùng biến report/clean_report đã có trong notebook; nếu chưa chạy cell trước thì dùng số liệu mẫu trong notebook.
import matplotlib.pyplot as plt

fallback_report = {
    "dataset_overview": {
        "images_total": 4578,
        "images_with_labels": 4578,
        "images_without_labels": 0,
        "total_boxes": 5200,
        "avg_boxes_per_image": 1.1358671909130624,
    },
    "plate_box_stats": {
        "avg_area_ratio": 0.04324780313333054,
        "min_area_ratio": 8.226653527747182e-05,
        "max_area_ratio": 0.7673624021100238,
    },
    "brightness_distribution": {"dark": 404, "normal": 4143, "bright": 31},
}

eda_report = report if "report" in globals() else fallback_report
overview = eda_report.get("dataset_overview", {})
box_stats = eda_report.get("plate_box_stats", {})
brightness = eda_report.get("brightness_distribution", {})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Label coverage
label_categories = ["Có nhãn", "Thiếu nhãn"]
label_values = [overview.get("images_with_labels", 0), overview.get("images_without_labels", 0)]
axes[0].bar(label_categories, label_values, color=["#59A14F", "#E15759"])
axes[0].set_title("Độ phủ nhãn")
axes[0].set_ylabel("Số ảnh")
for i, value in enumerate(label_values):
    axes[0].text(i, value, str(value), ha="center", va="bottom")

# 2) Brightness distribution
brightness_labels = list(brightness.keys())
brightness_values = list(brightness.values())
axes[1].bar(brightness_labels, brightness_values, color=["#4E79A7", "#76B7B2", "#F28E2B"])
axes[1].set_title("Phân bố độ sáng")
axes[1].set_ylabel("Số ảnh")
for i, value in enumerate(brightness_values):
    axes[1].text(i, value, str(value), ha="center", va="bottom")

# 3) Area ratio summary
area_labels = ["min", "avg", "max"]
area_values = [
    box_stats.get("min_area_ratio", 0),
    box_stats.get("avg_area_ratio", 0),
    box_stats.get("max_area_ratio", 0),
]
axes[2].bar(area_labels, area_values, color="#9C755F")
axes[2].set_title("Tỷ lệ diện tích bbox biển số")
axes[2].set_ylabel("area_ratio")
for i, value in enumerate(area_values):
    axes[2].text(i, value, f"{value:.4f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

print("Gợi ý đọc nhanh:")
print("- Thiếu nhãn = 0 là rất tốt cho bước train detector.")
print("- Brightness lệch mạnh về normal, nên cần balance/augmentation để giảm bias ánh sáng.")
print("- avg_area_ratio khoảng 0.043 nghĩa là biển số là object khá nhỏ trong ảnh.")

## Phần 17 - Kết quả Buổi 2 sẽ được đánh giá như thế nào?

Đánh giá Buổi 2 không phải xem model đúng/sai, mà là xem **dataset đã đủ tốt để train model chưa**.

Có thể đánh giá theo 5 nhóm.

### 1) Độ đầy đủ của dữ liệu và nhãn

Các chỉ số cần xem:

- `images_total`
- `images_with_labels`
- `images_without_labels`
- `total_boxes`
- `avg_boxes_per_image`

Công thức quan trọng:

$$
label\_coverage = \frac{images\_with\_labels}{images\_total}
$$

Ý nghĩa:

- `label_coverage` gần 1: hầu hết ảnh có nhãn, tốt cho train detector.
- `images_without_labels` cao: nhiều ảnh không thể dùng để train YOLO có giám sát.

Với số liệu hiện tại:

$$
label\_coverage = \frac{4578}{4578} = 1.0
$$

Đánh giá: độ phủ nhãn rất tốt vì không có ảnh thiếu nhãn.

### 2) Chất lượng bbox biển số

Các chỉ số cần xem:

- `avg_area_ratio`
- `min_area_ratio`
- `max_area_ratio`
- gallery ảnh preview bbox

Ý nghĩa:

- `avg_area_ratio` cho biết biển số chiếm trung bình bao nhiêu phần ảnh.
- `min_area_ratio` quá nhỏ có thể là biển rất xa hoặc label bất thường.
- `max_area_ratio` quá lớn có thể là bbox khoanh quá rộng hoặc ảnh crop quá sát.

Đánh giá bằng mắt:

- Bbox ôm đúng biển số: nhãn tốt.
- Bbox lệch: cần sửa label trước train.
- Bbox quá rộng: OCR sau này có thể đọc nhầm nền.
- Bbox quá hẹp: OCR sau này có thể mất ký tự.

### 3) Cân bằng điều kiện ánh sáng

Các chỉ số cần xem:

- `brightness_distribution.dark`
- `brightness_distribution.normal`
- `brightness_distribution.bright`

Một chỉ số đơn giản để nhìn độ lệch:

$$
imbalance\_ratio = \frac{\max(N_{dark}, N_{normal}, N_{bright})}{\min(N_{dark}, N_{normal}, N_{bright})}
$$

Với số liệu hiện tại:

$$
imbalance\_ratio = \frac{4143}{31} \approx 133.65
$$

Đánh giá: dữ liệu lệch rất mạnh về ảnh `normal`; nhóm `bright` rất ít. Nếu không xử lý, model có thể kém hơn khi gặp ảnh quá sáng hoặc quá tối.

Hành động phù hợp:

- Balance dữ liệu theo bucket sáng/tối.
- Augment tăng/giảm sáng.
- Bổ sung thêm ảnh trong điều kiện `dark` và `bright`.

### 4) Chất lượng augmentation

Augmentation chỉ tốt nếu ảnh và bbox biến đổi đồng bộ.

Cần kiểm tra gallery augmentation:

- Ảnh sáng/tối/rotate/resize có hợp lý không?
- Bbox sau biến đổi còn bám đúng biển số không?
- Có ảnh nào bị xoay/cắt làm mất biển số không?

Nếu bbox lệch sau augmentation, không nên dùng dữ liệu augment đó để train vì model sẽ học sai.

### 5) Chất lượng split train/val/test

Split tốt cần thỏa mãn:

- Có đủ `train.txt`, `val.txt`, `test.txt`.
- Tỉ lệ chia gần đúng cấu hình mong muốn.
- Không để cùng một ảnh xuất hiện ở nhiều split.
- Nên giữ phân bố ánh sáng/nguồn ảnh tương đối hợp lý giữa các split.

Ý nghĩa trong dự án:

- `train`: để model học.
- `val`: để theo dõi mô hình trong lúc train.
- `test`: để đánh giá cuối cùng công bằng hơn.

Nếu split sai, kết quả Buổi 3 có thể bị ảo: model nhìn như tốt nhưng thật ra test không khách quan.

## Kết luận mẫu cho báo cáo Buổi 2

Bạn có thể viết:

> Qua EDA, bộ dữ liệu có 4578 ảnh và toàn bộ ảnh đều có nhãn tương ứng, đạt độ phủ nhãn 100%. Tổng số bbox là 5200, trung bình khoảng 1.136 bbox mỗi ảnh. Tỷ lệ diện tích biển số trung bình khoảng 0.0432, cho thấy biển số là object tương đối nhỏ trong ảnh, cần chú ý khi train detector. Phân bố độ sáng bị lệch mạnh về nhóm `normal` với 4143 ảnh, trong khi nhóm `bright` chỉ có 31 ảnh, vì vậy bước cân bằng dữ liệu và augmentation ánh sáng là cần thiết. Sau khi kiểm tra preview bbox, clean/balance, augmentation và split train/val/test, dữ liệu đủ điều kiện chuyển sang Buổi 3 để train YOLO baseline.

## Tiêu chí quyết định đã sẵn sàng sang Buổi 3

Dataset được xem là sẵn sàng nếu:

- Không thiếu nhãn hoặc thiếu rất ít nhãn.
- Bbox preview đa số bám đúng biển số.
- Không còn ảnh lỗi/label rỗng nghiêm trọng.
- Đã có split `train/val/test`.
- Đã nhận diện được lệch phân bố chính và có xử lý bằng balance/augmentation.

Nếu chưa đạt các điểm trên, nên sửa dữ liệu trước khi train, vì model tốt không thể cứu một dataset sai nhãn hoặc lệch quá nặng.

---

## File code liên quan đến notebook Buổi 2

Các cell trong notebook này chủ yếu là phần trình bày và trực quan hóa. Logic chính nên xem ở các file code sau:

### 1) EDA dữ liệu

- [`../scripts/eda_dataset.py`](../scripts/eda_dataset.py): quét ảnh, đọc label YOLO, thống kê số ảnh, bbox, kích thước ảnh, độ sáng và sinh `reports/eda/dataset_report.json`.
- [`../reports/eda/dataset_report.json`](../reports/eda/dataset_report.json): file kết quả EDA mà notebook đọc lại để in số liệu và vẽ biểu đồ.
- [`../reports/eda/dataset_report.md`](../reports/eda/dataset_report.md): bản báo cáo EDA dạng Markdown.

### 2) Làm sạch và cân bằng dữ liệu

- [`../scripts/clean_balance_dataset.py`](../scripts/clean_balance_dataset.py): kiểm tra ảnh lỗi, label thiếu/sai, lọc dữ liệu sạch và cân bằng theo bucket sáng/tối.
- [`../reports/eda/clean_balance_report.json`](../reports/eda/clean_balance_report.json): kết quả clean/balance mà notebook dùng để giải thích.

### 3) Augmentation và tiền xử lý

- [`../scripts/preprocess_augment.py`](../scripts/preprocess_augment.py): tạo ảnh biến thể như resize, tăng sáng, xoay nhẹ và đồng bộ bbox.
- [`../src/preprocess/ops.py`](../src/preprocess/ops.py): các hàm xử lý crop/preprocess dùng lại trong pipeline inference.

### 4) Chia dữ liệu train/val/test

- [`../scripts/split_dataset.py`](../scripts/split_dataset.py): tạo các file split `train.txt`, `val.txt`, `test.txt`.
- [`../scripts/build_manifest.py`](../scripts/build_manifest.py): tạo manifest dữ liệu nếu cần quản lý danh sách ảnh/nhãn rõ hơn.

### 5) Nên đọc theo thứ tự

1. Notebook này để hiểu ý nghĩa số liệu.
2. [`../scripts/eda_dataset.py`](../scripts/eda_dataset.py) để hiểu số liệu được tính ra sao.
3. [`../scripts/clean_balance_dataset.py`](../scripts/clean_balance_dataset.py) để hiểu dữ liệu được lọc/cân bằng thế nào.
4. [`../scripts/preprocess_augment.py`](../scripts/preprocess_augment.py) để hiểu ảnh và bbox được biến đổi ra sao.